In [1]:
# %pip install transformers[torch] accelerate datasets scikit-learn -U

import pandas as pd
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from accelerate.utils import find_executable_batch_size

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Sample dataset
data = pd.DataFrame({
    "text": [
        "  The staff was very kind and attentive to my needs!!!  ",
        "The waiting time was too long, and the staff was rude. Visit us at http://hospitalreviews.com",
        "The doctor answered all my questions...but the facility was outdated.   ",
        "The nurse was compassionate & made me feel comfortable!! :) ",
        "I had to wait over an hour before being seen.  Unacceptable service! #frustrated",
        "The check-in process was smooth, but the doctor seemed rushed. Visit https://feedback.com",
        "Everyone I interacted with was professional and helpful.  "
    ],
    "label": ["positive", "negative", "neutral", "positive", "negative", "neutral", "positive"]
})

# Clean text
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()

data["cleaned_text"] = data["text"].apply(clean_text)

# Encode labels
label_map = {"positive": 0, "neutral": 1, "negative": 2}
data["label"] = data["label"].map(label_map)

# Split data
train_data, temp_data = train_test_split(data, test_size=0.3, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

# Hugging Face Dataset conversion
train_ds = Dataset.from_pandas(train_data)
val_ds = Dataset.from_pandas(val_data)
test_ds = Dataset.from_pandas(test_data)

# Tokenization function
def tokenize_fn(batch):
    return tokenizer(batch["cleaned_text"], padding="max_length", truncation=True, max_length=128)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

# Remove non-tensor columns
cols_to_remove = ["text", "cleaned_text", "__index_level_0__"]
train_ds = train_ds.remove_columns([c for c in cols_to_remove if c in train_ds.column_names])
val_ds = val_ds.remove_columns([c for c in cols_to_remove if c in val_ds.column_names])
test_ds = test_ds.remove_columns([c for c in cols_to_remove if c in test_ds.column_names])

# Ensure label is int
train_ds = train_ds.map(lambda x: {"label": int(x["label"])})
val_ds = val_ds.map(lambda x: {"label": int(x["label"])})
test_ds = test_ds.map(lambda x: {"label": int(x["label"])})

# Load model
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)

@find_executable_batch_size(starting_batch_size=16)
def get_training_args(batch_size):
    return TrainingArguments(
        output_dir="./results",
        learning_rate=2e-5,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=3,
        eval_strategy="steps",  # Use eval_strategy parameter
        eval_steps=500,
        logging_dir="./logs",
        logging_steps=100,
        save_strategy="steps",
        save_steps=500,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        gradient_accumulation_steps=max(1, 16 // batch_size),
        fp16=False,
        weight_decay=0.01
    )

training_args = get_training_args()

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds.with_format("torch", columns=["input_ids", "attention_mask", "label"]),
    eval_dataset=val_ds.with_format("torch", columns=["input_ids", "attention_mask", "label"])
)

# Train
trainer.train

# Evaluate
predictions = trainer.predict(test_ds.with_format("torch"))
preds = predictions.predictions.argmax(-1)
labels = test_ds["label"]

acc = accuracy_score(labels, preds)
f1 = f1_score(labels, preds, average="weighted")
print(f"Accuracy: {acc:.4f}, F1 Score: {f1:.4f}")

# Save model
model.save_pretrained("./fine_tuned_bert")
tokenizer.save_pretrained("./fine_tuned_bert")
print("Model saved successfully.")


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\Jamie\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Accuracy: 0.5000, F1 Score: 0.3333
Model saved successfully.
